In [8]:
!pip install --force-reinstall numpy==1.26.4

  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


In [30]:
import numpy
print(numpy.__version__)

2.2.6


In [10]:
import pandas as pd
import numpy as np
import random
# import pyspark.pandas as ps
from pyspark.sql import SparkSession

In [11]:
existing_spark = SparkSession.getActiveSession()

if existing_spark:
    print("Existing Spark session found. Stopping it...")
    existing_spark.stop()

In [12]:
spark = (
        SparkSession.builder
        .appName("DataFrames App")
        .master("local[*]")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,"
                                        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
                                        "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0,"
                                        "org.apache.hadoop:hadoop-aws:3.3.4,"
                                        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
                                        "software.amazon.awssdk:glue:2.25.26,"
                                        "software.amazon.awssdk:sts:2.25.26,"
                                        "software.amazon.awssdk:s3:2.25.26,"
                                        "software.amazon.awssdk:dynamodb:2.25.26,"
                                        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
                                        "software.amazon.awssdk:url-connection-client:2.25.26")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,"
                                        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
                                        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
        .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.warehouse.dir", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.catalog.glue_catalog.client.region", "ap-south-1")
        .config("spark.sql.defaultCatalog", "spark_catalog")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("hive.metastore.client.factory.class", "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
        .enableHiveSupport()
        .getOrCreate()
    )

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hudi#hudi-spark3.5-bundle_2.12 added as a dependency
software.amazon.awssdk#glue added as a dependency
software.amazon.awssdk#sts added as a dependency
software.amazon.awssdk#s3 added as a dependency
software.amazon.awssdk#url-connection-client added as a dependency
software.amazon.awssdk#dynamodb added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-21857d61-6ca8-4d3a-946d-0687ea122b6c;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.a

In [13]:
spark

26/04/28 11:43:13 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [14]:
data = [(i, "user_" + str(i), random.randint(18, 60)) for i in range(30)]

In [15]:
user = spark.sparkContext.parallelize(data)

In [16]:
user.collect()

[(0, 'user_0', 26),
 (1, 'user_1', 27),
 (2, 'user_2', 23),
 (3, 'user_3', 50),
 (4, 'user_4', 28),
 (5, 'user_5', 40),
 (6, 'user_6', 54),
 (7, 'user_7', 37),
 (8, 'user_8', 53),
 (9, 'user_9', 51),
 (10, 'user_10', 58),
 (11, 'user_11', 25),
 (12, 'user_12', 44),
 (13, 'user_13', 47),
 (14, 'user_14', 53),
 (15, 'user_15', 59),
 (16, 'user_16', 55),
 (17, 'user_17', 46),
 (18, 'user_18', 29),
 (19, 'user_19', 24),
 (20, 'user_20', 57),
 (21, 'user_21', 30),
 (22, 'user_22', 48),
 (23, 'user_23', 33),
 (24, 'user_24', 41),
 (25, 'user_25', 54),
 (26, 'user_26', 50),
 (27, 'user_27', 54),
 (28, 'user_28', 43),
 (29, 'user_29', 25)]

In [17]:
user_df = user.toDF()

In [18]:
user_df.show()

+---+-------+---+
| _1|     _2| _3|
+---+-------+---+
|  0| user_0| 26|
|  1| user_1| 27|
|  2| user_2| 23|
|  3| user_3| 50|
|  4| user_4| 28|
|  5| user_5| 40|
|  6| user_6| 54|
|  7| user_7| 37|
|  8| user_8| 53|
|  9| user_9| 51|
| 10|user_10| 58|
| 11|user_11| 25|
| 12|user_12| 44|
| 13|user_13| 47|
| 14|user_14| 53|
| 15|user_15| 59|
| 16|user_16| 55|
| 17|user_17| 46|
| 18|user_18| 29|
| 19|user_19| 24|
+---+-------+---+
only showing top 20 rows



In [19]:
df_user = user.toDF(['id', 'username', 'age'])

In [20]:
df_user.show()

+---+--------+---+
| id|username|age|
+---+--------+---+
|  0|  user_0| 26|
|  1|  user_1| 27|
|  2|  user_2| 23|
|  3|  user_3| 50|
|  4|  user_4| 28|
|  5|  user_5| 40|
|  6|  user_6| 54|
|  7|  user_7| 37|
|  8|  user_8| 53|
|  9|  user_9| 51|
| 10| user_10| 58|
| 11| user_11| 25|
| 12| user_12| 44|
| 13| user_13| 47|
| 14| user_14| 53|
| 15| user_15| 59|
| 16| user_16| 55|
| 17| user_17| 46|
| 18| user_18| 29|
| 19| user_19| 24|
+---+--------+---+
only showing top 20 rows



In [21]:
cols = ("ID", "Username", "Age")
df_users = df_user.toDF(*cols)

In [22]:
df_users.show(5)

+---+--------+---+
| ID|Username|Age|
+---+--------+---+
|  0|  user_0| 26|
|  1|  user_1| 27|
|  2|  user_2| 23|
|  3|  user_3| 50|
|  4|  user_4| 28|
+---+--------+---+
only showing top 5 rows



In [23]:
df_user.show(5)

+---+--------+---+
| id|username|age|
+---+--------+---+
|  0|  user_0| 26|
|  1|  user_1| 27|
|  2|  user_2| 23|
|  3|  user_3| 50|
|  4|  user_4| 28|
+---+--------+---+
only showing top 5 rows



In [24]:
pdf = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Cathy"],
    "age": [25, 30, 28]
})

In [25]:
pdf_df = spark.createDataFrame(pdf)

In [26]:
pdf_df.show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 25|
|  2|  Bob| 30|
|  3|Cathy| 28|
+---+-----+---+



In [27]:
pd_pdf_df = pdf_df.toPandas()

In [28]:
pd_pdf_df.head()

,id,name,age
0,1,Alice,25
1,2,Bob,30
2,3,Cathy,28


In [17]:
df = ps.from_pandas(pd_pdf_df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

/home/glue_user/spark/python/pyspark/pandas/internal.py:1573: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  fields = [
/home/glue_user/spark/python/pyspark/sql/pandas/conversion.py:486: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():

In [18]:
df.head()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

   id   name  age
0   1  Alice   25
1   2    Bob   30
2   3  Cathy   28

In [29]:
spark.stop()